# BSAN391 - Minimum Cover EMS - Set covering - PuLP

The city of Austin, Texas (source: Optimization in Operations Research, by Rardin) undertook a study of the positioning of its emergency medical service (EMS) vehicles. The city was divided into service districts needing EMS services, and vehicle stations selected from a list of alternatives so that as much of the population as possible would experience a quick response to calls for help. The figure below shows the fictitious map we will assume for our numerical version. Our city is divided into 20 service districts that we wish to serve from some combination of the 10 indicated possibilities for EMS stations. Each station can provide service to all $\textbf{adjacent}$ districts.

<img src="ems.png" style="width:280px;height:250px"/>

In [1]:
# Import PuLP
from pulp import *

In [2]:
# N districts
N = 20
Districts = [i for i in range(1,N+1)]

# M stations
M = 10
Stations = [j for j in range(1,M+1)]

# Adjacency
A = {1:{2}, 2:{1,2}, 3:{1,3}, 4:{3}, 5:{3}, 6:{2}, 7:{2,4}, 8:{3,4}, 9:{8}, 10:{4,6},11:{4,5}, 12:{4,5,6},
     13:{4,5,7}, 14:{8,9}, 15:{6,9}, 16:{5,6}, 17:{5,7,10}, 18:{8,9}, 19:{9,10}, 20:{10}}

In [3]:
#Problem definition
prob = LpProblem("EMS_cover",LpMinimize)

In [4]:
# Decision Variables (binary variables, 1 if station is selected, 0 otherwise)
x = LpVariable.dicts("Station",Stations,0,1,LpInteger)

In [5]:
# Objective Function
prob += lpSum([x[j] for j in Stations]), "Number_of_opened_stations"

In [6]:
# Covering constraints
for i in Districts:
    prob+= lpSum([x[j] for j in A[i]]) >= 1, "District %s"%i

In [7]:
# Solving IP
prob.solve()
print("Status:", LpStatus[prob.status])

Status: Optimal


In [8]:
#Print optimal objective function value
print("Minimum_number_of_opened_stations=", str(value(prob.objective)))

Minimum_number_of_opened_stations= 6.0


In [9]:
#Print optimal solution - selected locations
for variable in prob.variables():
    if variable.varValue == 1:
        print ("{}* = {}".format(variable.name, variable.varValue))

Station_10* = 1.0
Station_2* = 1.0
Station_3* = 1.0
Station_4* = 1.0
Station_6* = 1.0
Station_8* = 1.0


In the Austin case, as in many other real instances, the straightforward covering model above proves inadequate because it calls for too many sites. Suppose that we have funds for only 4 EMS locations. How can we find the collection of 4 that $\textbf{minimizes coverage insufficiency}$?
For this version of the model we need estimates of the demand or importance of covering each service district. We will assume the following values have been estimated by EMS staff:

In [10]:
V = {1: 5.2, 2: 4.4, 3: 7.1, 4: 9.0, 5: 6.1, 6: 5.7, 7: 10.0, 8: 12.2, 9: 7.6, 10: 20.3, 11: 20.3, 12: 30.9,
     13: 12.0, 14: 9.3, 15: 15.5, 16: 25.6, 17: 11.0, 18: 5.3, 19: 7.9, 20: 9.9}

For the sake of clarity, we'll define a brand new problem. Next we introduce extra decision variables to model uncovered districts $i$:

In [11]:
#New Problem definition
prob2 = LpProblem("Maximum_coverage",LpMinimize)

In [12]:
xnew = LpVariable.dicts("Station",Stations,0,1,LpInteger)
# Decision Variables (binary variables, 1 if district is uncovered, 0 otherwise)
y = LpVariable.dicts("Uncovered",Districts,0,1,LpInteger)

In [13]:
# New Objective Function
prob2 += lpSum([(V[i]*y[i]) for i in Districts]), "Uncovered_distance_importance"

In [14]:
# New Covering constraints
for i in Districts:
    prob2+= lpSum([xnew[j] for j in A[i]]) + y[i] >= 1, "District %s"%i

In [15]:
# At most four stations
prob2 += lpSum([xnew[j] for j in Stations]) <= 4, "Max_number_of_opened_stations"
#prob2

In [16]:
# Solving IP
prob2.solve()
print("Status:", LpStatus[prob2.status])

Status: Optimal


In [17]:
#Print optimal objective function value
print("Minimum_uncovered_district_importance=", str(value(prob2.objective)))

Minimum_uncovered_district_importance= 32.8


In [18]:
#Print optimal solution
for variable in prob2.variables():
    if variable.varValue == 1:
        print ("{}* = {}".format(variable.name, variable.varValue))

Station_3* = 1.0
Station_4* = 1.0
Station_5* = 1.0
Station_9* = 1.0
Uncovered_1* = 1.0
Uncovered_2* = 1.0
Uncovered_20* = 1.0
Uncovered_6* = 1.0
Uncovered_9* = 1.0
